GOLD LAYER — CUSTOMER ENGAGEMENT 

Table: gold.customer_engagement
Grain: 1 row = 1 customer

Business goal:

Measure customer activity
Compare engagement vs revenue
Identify active vs passive customers

1. Load Tables

In [0]:
df_interactions = spark.table("silver.crm_interactions")
df_sales = spark.table("silver.sales_orders")
df_customers = spark.table("silver.crm_customers")

2. Aggregate Interactions

In [0]:
from pyspark.sql.functions import count, sum, when

df_interactions_agg = df_interactions.groupBy("customer_id").agg(
    count("*").alias("total_interactions"),
    
    sum(when(df_interactions.interaction_type == "website_visit", 1).otherwise(0)).alias("total_visits"),
    
    sum(when(df_interactions.interaction_type == "support_ticket", 1).otherwise(0)).alias("total_support_tickets")
)

3. Aggregate Sales

In [0]:
from pyspark.sql.functions import sum, countDistinct

df_sales_agg = df_sales.groupBy("customer_id").agg(
    sum("total_amount").alias("total_revenue"),
    countDistinct("order_id").alias("total_orders")
)

4. Join Interactions + Sales

In [0]:
df_engagement = df_interactions_agg.join(
    df_sales_agg,
    on="customer_id",
    how="left"
)

5. Add Conversion Metric

In [0]:
from pyspark.sql.functions import round

df_engagement = df_engagement.withColumn(
    "orders_per_visit",
    round(df_engagement.total_orders / df_engagement.total_interactions, 4)
)

6. Join Customer Attributes

In [0]:
from pyspark.sql.functions import when, round, col

df_engagement = df_engagement.withColumn(
    "orders_per_visit",
    when(col("total_visits") > 0,
         round(col("total_orders") / col("total_visits"), 2)
    ).otherwise(0)
)

In [0]:
from pyspark.sql.functions import when, col

df_engagement = df_engagement.withColumn(
    "orders_per_visit",
    when(col("orders_per_visit") > 10, 10).otherwise(col("orders_per_visit"))
)

engagement_level

In [0]:
from pyspark.sql.functions import when

df_engagement = df_engagement.withColumn(
    "engagement_level",
    when(
        (col("total_interactions") >= 15) & (col("total_orders") >= 20),
        "High"
    ).when(
        (col("total_interactions") >= 8) & (col("total_orders") >= 10),
        "Medium"
    ).otherwise("Low")
)

6. Join Customer Attributes

In [0]:
df_engagement = df_engagement.join(
    df_customers.select("customer_id", "customer_name", "loyalty_tier"),
    on="customer_id",
    how="left"
)

7. Select Final Columns

In [0]:
df_engagement = df_engagement.select(
    "customer_id",
    "customer_name",
    "loyalty_tier",
    "total_interactions",
    "total_visits",
    "total_support_tickets",
    "total_orders",
    "total_revenue",
    "orders_per_visit",
    "engagement_level"
)

8. Write to Gold Table

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("DROP TABLE IF EXISTS gold.customer_engagement")

df_engagement.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.customer_engagement")

9. Validation

In [0]:
display(spark.table("gold.customer_engagement"))